# ReasonIF Prefix-Intervention Evaluation: Untouched Phi-4 Base + Constraint-ON Prefix

This notebook evaluates whether injecting the first **10 reasoning tokens** from the matched constraint_on reasoning trace into the `<think>` block of the **untouched base `microsoft/Phi-4-reasoning`** model unlocks latent instruction-following controllability on the official 300 ReasonIF benchmark problems.

### Dual Evaluation Metrics
1. **Official IFS (`official_instruction_following`)**: Evaluates the entire reasoning trace (`forced_10_tokens + continuation`).
2. **Continuation IFS (`continuation_instruction_following`)**: Evaluates strictly the continuation generated by base Phi-4 (excluding the forced 10 tokens), isolating whether base Phi-4's own generated tokens adhered to the constraint.
3. **Math Accuracy & Joint Success**: Exact canonical math verification on the final answer, plus joint success under both metrics.
4. **Paired Statistical Comparison**: Side-by-side McNemar test against untouched base Phi-4 (baseline: 5.0% IFS, 75.3% accuracy).


## 1. Verify the GPU and environment

Requires an NVIDIA GPU with at least 40 GB VRAM (e.g. A100, H100, RTX PRO 6000 Blackwell) and `vllm==0.16.0`.

In [ ]:
import sys
print('Python executable:', sys.executable)
print('Python version:', sys.version)


In [ ]:
# Ensure FlashInfer sampling JIT conflicts are disabled on Blackwell
import os
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
os.environ["CUDA_HOME"] = "/usr/local/cuda"

import torch
assert torch.cuda.is_available(), 'Attach an NVIDIA GPU before continuing.'
gpu_name = torch.cuda.get_device_name(0)
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print('GPU:', gpu_name, f'({gpu_gib:.1f} GiB)')
assert gpu_gib >= 38, 'Native-BF16 Phi-4-14B inference needs approximately 40 GB VRAM or more.'
print('Environment verification complete.')


In [ ]:
!nvidia-smi


## 2. Configuration: Untouched Phi-4 Base with Constraint-OFF Prefix Injection

We evaluate all **300 official ReasonIF benchmark questions** under `bfloat16` precision using vLLM.
The model generates inside `<think>` with pre-filled 10-token unconstrained planning prefix tokens.


In [ ]:
# ===================== EVALUATION CONFIGURATION =====================
EVAL_MODE = "full"
LOAD_MODEL_ID = "microsoft/Phi-4-reasoning"
PREFIX_CONDITION = "constraint_on"  # 'constraint_on' or 'constraint_off'
FORCED_TOKENS_K = 10
MODEL_LABEL = "phi4_reasoning_14b_base_prefix_constraint_on"
MODEL_ID = LOAD_MODEL_ID
MODEL_REVISION = None
LOAD_MODEL_REVISION = None
OUTPUT_BASE = None

NUM_QUESTIONS = 300
REASONIF_COMMIT = "706b953feb9408a802e1ad6972c10ad7fbad3da8"
REASONIF_MODEL_KEY = "Qwen3-14B"  # Pinned scale calibration baseline
GENERATION_SEED = 42
MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 16384
MAX_MODEL_LEN = MAX_INPUT_TOKENS + MAX_NEW_TOKENS
TEMPERATURE = 1.0
TOP_P = 0.95
CHUNK_SIZE = 32
MAX_NUM_SEQS = 32
GPU_MEMORY_UTILIZATION = 0.90

assert NUM_QUESTIONS == 300
assert MAX_MODEL_LEN == 18432
assert TEMPERATURE == 1.0 and TOP_P == 0.95 and GENERATION_SEED == 42
print(f"Evaluation mode: {EVAL_MODE}")
print(f"Base model: {LOAD_MODEL_ID}")
print(f"Prefix condition: {PREFIX_CONDITION} (forced k={FORCED_TOKENS_K} tokens)")
print("Protocol: BF16, temp=1.0, top_p=0.95, max_new_tokens=16384, all 300 questions")

## 3. Pin the model and benchmark; prepare the official 300 prompts

In [ ]:
import hashlib, json, os, platform, re, shutil, subprocess, sys, time
from pathlib import Path
from huggingface_hub import HfApi, snapshot_download

LAUNCH_DIR = Path.cwd().resolve()
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

REASONIF_ROOT = LAUNCH_DIR / 'reasonIF_pinned_final_eval'
if not (REASONIF_ROOT / '.git').is_dir():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ykwon0407/reasonIF.git', str(REASONIF_ROOT)], check=True)
subprocess.run(['git', 'checkout', '--quiet', REASONIF_COMMIT], cwd=REASONIF_ROOT, check=True)
actual_reasonif_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REASONIF_ROOT, text=True).strip()
assert actual_reasonif_commit == REASONIF_COMMIT
if str(REASONIF_ROOT) not in sys.path:
    sys.path.insert(0, str(REASONIF_ROOT))
from src.utils import prepare_message_list

safe_model = re.sub(r'[^a-z0-9]+', '_', MODEL_LABEL.lower()).strip('_')
run_key = f'{EVAL_MODE}_prefix_{PREFIX_CONDITION}_k{FORCED_TOKENS_K}_n{NUM_QUESTIONS}_gseed{GENERATION_SEED}_new{MAX_NEW_TOKENS}_temp1_top_p095_native_chat'
base = Path(OUTPUT_BASE).expanduser().resolve() if OUTPUT_BASE else LAUNCH_DIR / 'outputs' / 'reasonif_phi4_prefix_intervention'
RESULTS_ROOT = base / safe_model / run_key
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
RAW_PATH = RESULTS_ROOT / 'raw_responses.jsonl'

# Fix calibration path: Ensure official word limit reference is loaded correctly
orig_cwd = os.getcwd()
try:
    os.chdir(str(REASONIF_ROOT))
    dataset_path = Path("data") / "reasonIF_dataset.json"
    official_messages, official_dataset = prepare_message_list(
        REASONIF_MODEL_KEY, input_path=str(dataset_path)
    )
finally:
    os.chdir(orig_cwd)

assert len(official_messages) == len(official_dataset) == NUM_QUESTIONS
selected_indices = list(range(NUM_QUESTIONS))
selected_rows = [dict(official_dataset[index], dataset_index=index) for index in selected_indices]
(RESULTS_ROOT / 'reasonif_dataset_after_official_preparation.json').write_text(
    json.dumps(selected_rows, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)

api = HfApi()
base_info = api.model_info(LOAD_MODEL_ID, token=HF_TOKEN)
resolved_load_revision = LOAD_MODEL_REVISION or base_info.sha
resolved_revision = resolved_load_revision

# Resolve Phi-4 SFT donor path (Constraint-ON)
donor_candidates = [
    LAUNCH_DIR / 'outputs' / 'reasonif_phi4_reasoning_final_paper' / 'phi4_reasoning_14b_gpt52_high' / 'lora_n300_gseed42_new16384_temp1_top_p095_native_chat' / 'scored_responses.jsonl',
    Path('/home/kyleliu789/uncontaminated_workspace/reasonif/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl'),
    Path('/home/kyleliu789/workspace/outputs/reasonif_phi4_reasoning_final_paper/phi4_reasoning_14b_gpt52_high/lora_n300_gseed42_new16384_temp1_top_p095_native_chat/scored_responses.jsonl'),
    LAUNCH_DIR.parent / 'outputs' / 'reasonif_phi4_reasoning_final_paper' / 'phi4_reasoning_14b_gpt52_high' / 'lora_n300_gseed42_new16384_temp1_top_p095_native_chat' / 'scored_responses.jsonl',
]
PREFIX_SOURCE_PATH = None
for cand in donor_candidates:
    if cand.is_file():
        PREFIX_SOURCE_PATH = cand
        break
assert PREFIX_SOURCE_PATH is not None, f"Phi-4 SFT donor file not found among candidates: {donor_candidates}"
DONOR_EXPECTED_MODEL = "kyleliu789/phi4-reasoning-14b-gpt52-high-reasoning-original"


manifest = {
    'evaluation_mode': EVAL_MODE,
    'model_id': MODEL_ID,
    'model_revision': resolved_revision,
    'load_model_id': LOAD_MODEL_ID,
    'load_model_revision': resolved_load_revision,
    'prefix_condition': PREFIX_CONDITION,
    'donor_expected_model': DONOR_EXPECTED_MODEL,
    'prefix_source_path': str(PREFIX_SOURCE_PATH),
    'forced_tokens_k': FORCED_TOKENS_K,
    'reasonif_commit': actual_reasonif_commit,
    'num_questions': NUM_QUESTIONS,
    'generation_seed_rule': '42 + dataset_index',
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'max_input_tokens': MAX_INPUT_TOKENS,
    'max_new_tokens': MAX_NEW_TOKENS,
    'max_model_len': MAX_MODEL_LEN,
    'dtype': 'bfloat16',
    'python': platform.python_version(),
}
manifest_path = RESULTS_ROOT / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('Prefix source verified:', PREFIX_SOURCE_PATH)
print('Frozen loaded-model revision:', resolved_load_revision)
print('Results root:', RESULTS_ROOT)

## 4. Extract forced 10-token reasoning prefixes and initialize vLLM engine

For each prompt, we take the native Qwen chat prompt (which ends at `<|im_start|>assistant\n<think>\n`) and append the exact 10 tokenizer token IDs from the reference trace. The model is then invoked via `prompt_token_ids`, forcing it to begin its reasoning with those exact 10 tokens.

In [ ]:
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
import vllm

tokenizer = AutoTokenizer.from_pretrained(
    LOAD_MODEL_ID, revision=resolved_load_revision, token=HF_TOKEN, trust_remote_code=True,
)

def render_native_chat(messages):
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, 
    )

# Load and strictly verify donor trace
with open(PREFIX_SOURCE_PATH, 'r', encoding='utf-8') as f:
    prefix_source_rows = [json.loads(line) for line in f if line.strip()]
assert len(prefix_source_rows) == NUM_QUESTIONS, f'Expected {NUM_QUESTIONS} prefix rows, got {len(prefix_source_rows)}'

# Map donor rows by dataset_index to verify 1:1 question alignment
donor_map = {row.get('dataset_index', row.get('prompt_id')): row for row in prefix_source_rows}
for index in range(NUM_QUESTIONS):
    assert index in donor_map, f"Missing donor row for dataset index {index}"
    d_row = donor_map[index]
    # Verify donor model identity
    if PREFIX_CONDITION == 'constraint_on':
        assert d_row.get('model_id') == DONOR_EXPECTED_MODEL, (
            f"Donor model mismatch at index {index}: {d_row.get('model_id')} vs {DONOR_EXPECTED_MODEL}"
        )
        assert d_row.get('question', '').strip() == official_dataset[index]['question'].strip(), (
            f"Donor question mismatch at dataset index {index}"
        )
    else:
        assert d_row.get('model') == "gpt52_short", (
            f"Donor model mismatch at index {index}: {d_row.get('model')} vs gpt52_short"
        )
        assert d_row.get('constraint_mode') == "constraint_off"
        q_target = official_dataset[index]['question'].strip()
        donor_prompt = d_row.get('prompt', '')
        assert q_target[:35] in donor_prompt or d_row.get('question', '').strip() == q_target, (
            f"Donor question mismatch at dataset index {index}"
        )

print(f"Verified all {NUM_QUESTIONS} donor rows (model: {DONOR_EXPECTED_MODEL}, aligned 1:1 with official questions).")

rendered_prompts = [render_native_chat(messages) for messages in official_messages]
base_prompt_token_ids = [tokenizer(prompt, add_special_tokens=False)['input_ids'] for prompt in rendered_prompts]

# Explicit <think> tag opening token so forced prefix is strictly inside the reasoning block
think_token_ids = tokenizer.encode('<think>', add_special_tokens=False)

forced_prefix_token_ids = []
forced_prefix_texts = []
full_prompt_token_ids = []

for index in range(NUM_QUESTIONS):
    rc = donor_map[index].get('reasoning_content', '').strip()
    f_ids = tokenizer.encode(rc, add_special_tokens=False)[:FORCED_TOKENS_K]
    f_text = tokenizer.decode(f_ids)
    forced_prefix_token_ids.append(f_ids)
    forced_prefix_texts.append(f_text)
    # Explicitly place <think> before the injected prefix tokens
    full_ids = base_prompt_token_ids[index] + think_token_ids + f_ids
    if len(full_ids) > MAX_INPUT_TOKENS:
        raise ValueError(f'Input exceeds {MAX_INPUT_TOKENS} tokens at dataset index {index}: {len(full_ids)}')
    full_prompt_token_ids.append(full_ids)

prompt_audit = [
    {
        'dataset_index': index,
        'base_input_tokens': len(base_prompt_token_ids[index]),
        'think_tag_tokens': len(think_token_ids),
        'forced_prefix_tokens': len(forced_prefix_token_ids[index]),
        'total_input_tokens': len(full_prompt_token_ids[index]),
        'forced_prefix_token_ids': forced_prefix_token_ids[index],
        'forced_prefix_text': forced_prefix_texts[index],
        'rendered_prompt_sha256': hashlib.sha256(rendered_prompts[index].encode('utf-8')).hexdigest(),
    }
    for index in range(NUM_QUESTIONS)
]
(RESULTS_ROOT / 'prompt_audit.json').write_text(json.dumps(prompt_audit, indent=2) + '\n', encoding='utf-8')
manifest['chat_template_sha256'] = hashlib.sha256((tokenizer.chat_template or '').encode('utf-8')).hexdigest()
manifest['maximum_observed_input_tokens'] = max(map(len, full_prompt_token_ids))

print(f'Sample index 0 forced prefix ({len(forced_prefix_token_ids[0])} tokens): {forced_prefix_token_ids[0]} -> {repr(forced_prefix_texts[0])}')

llm_kwargs = dict(
    model=LOAD_MODEL_ID, revision=resolved_load_revision, tokenizer=LOAD_MODEL_ID,
    tokenizer_revision=resolved_load_revision, dtype='bfloat16', quantization=None,
    trust_remote_code=True, max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION, max_num_seqs=MAX_NUM_SEQS,
    seed=GENERATION_SEED,
)
llm = LLM(**llm_kwargs)
manifest['vllm_version'] = vllm.__version__
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
print('vLLM engine initialized:', vllm.__version__)

## 5. Generate continuations in resumable, atomically saved chunks

Since the 10 forced tokens are placed inside `<think>`, the model generates the rest of the reasoning trace, closes `</think>`, and outputs `<answer>...</answer>`.

In [ ]:
from tqdm.auto import tqdm

def split_reasoning(raw_output):
    text = raw_output.strip()
    if '</think>' in text:
        reasoning, content = text.split('</think>', 1)
        return reasoning.removeprefix('<think>').strip(), content.strip()
    if text.startswith('<think>'):
        return text[len('<think>'):].strip(), ''
    return text, ''

def read_and_validate_jsonl(path, expected_manifest):
    completed = {}
    if not Path(path).is_file():
        return completed
    
    # Validate run configuration before resuming
    manifest_file = Path(path).parent / 'run_manifest.json'
    if manifest_file.is_file():
        try:
            existing_manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
            for key in ['model_id', 'load_model_id', 'prefix_condition', 'forced_tokens_k', 'temperature', 'top_p', 'max_new_tokens']:
                if key in existing_manifest and key in expected_manifest:
                    assert existing_manifest[key] == expected_manifest[key], (
                        f"Resume aborted: manifest mismatch on {key}: {existing_manifest[key]} vs {expected_manifest[key]}"
                    )
        except AssertionError:
            raise
        except Exception as e:
            print(f"Warning: Manifest check encounter: {e}")
    
    for line_number, line in enumerate(Path(path).read_text(encoding='utf-8').split('\n'), 1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(f'Invalid JSONL at {path}:{line_number}: {error}') from error
        idx = row['dataset_index']
        assert row.get('prefix_condition') == expected_manifest['prefix_condition'], (
            f"Resume mismatch: row prefix_condition {row.get('prefix_condition')} != {expected_manifest['prefix_condition']}"
        )
        assert row.get('forced_tokens_k') == expected_manifest['forced_tokens_k'], (
            f"Resume mismatch: row forced_tokens_k {row.get('forced_tokens_k')} != {expected_manifest['forced_tokens_k']}"
        )
        completed[idx] = row
    return completed

def append_records_atomic(path, records):
    path = Path(path)
    existing = path.read_bytes() if path.exists() else b''
    addition = ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in records).encode('utf-8')
    temporary = path.with_suffix(path.suffix + f'.{os.getpid()}.tmp')
    with temporary.open('wb') as sink:
        sink.write(existing); sink.write(addition); sink.flush(); os.fsync(sink.fileno())
    os.replace(temporary, path)

completed = read_and_validate_jsonl(RAW_PATH, manifest)
pending = [row for row in selected_rows if row['dataset_index'] not in completed]
progress = tqdm(total=NUM_QUESTIONS, initial=len(completed), desc='ReasonIF Prefix Gen')

for start in range(0, len(pending), CHUNK_SIZE):
    chunk_rows = pending[start:start + CHUNK_SIZE]
    chunk_indices = [row['dataset_index'] for row in chunk_rows]
    chunk_token_prompts = [{'prompt_token_ids': full_prompt_token_ids[idx]} for idx in chunk_indices]
    chunk_params = [
        SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_NEW_TOKENS,
                       seed=GENERATION_SEED + idx)
        for idx in chunk_indices
    ]
    outputs = llm.generate(
        chunk_token_prompts, chunk_params, lora_request=None, use_tqdm=True,
    )
    records = []
    for row, request_output in zip(chunk_rows, outputs):
        candidate = request_output.outputs[0]
        raw_continuation = candidate.text
        idx = row['dataset_index']
        forced_prefix = forced_prefix_texts[idx]
        forced_ids = forced_prefix_token_ids[idx]
        gen_ids = list(candidate.token_ids)
        
        # Token-level full reconstruction without artificial spaces
        full_thought_ids = forced_ids + gen_ids
        raw_full_thought = tokenizer.decode(full_thought_ids, skip_special_tokens=False)
        
        continuation_reasoning, content = split_reasoning(raw_continuation)
        full_reasoning_content, _ = split_reasoning(raw_full_thought)
        full_raw_output = f'<think>\n{full_reasoning_content}\n</think>\n{content}'
        
        records.append({
            **row,
            'model_id': MODEL_ID,
            'model_revision': resolved_revision,
            'evaluation_mode': EVAL_MODE,
            'prefix_condition': PREFIX_CONDITION,
            'forced_tokens_k': FORCED_TOKENS_K,
            'forced_prefix_text': forced_prefix,
            'forced_prefix_token_ids': forced_ids,
            'base_model_id': LOAD_MODEL_ID,
            'base_model_revision': resolved_load_revision,
            'raw_output': full_raw_output,
            'raw_continuation': raw_continuation,
            'reasoning_content': full_reasoning_content,
            'continuation_reasoning_content': continuation_reasoning.strip(),
            'content': content,
            'prompt_tokens': len(request_output.prompt_token_ids),
            'output_tokens': len(gen_ids),
            'finish_reason': candidate.finish_reason,
            'truncated': candidate.finish_reason == 'length',
            'seed': GENERATION_SEED + row['dataset_index'],
        })
    append_records_atomic(RAW_PATH, records)
    completed.update({row['dataset_index']: row for row in records})
    progress.update(len(records))

progress.close()
print(f'Completed: {len(completed)}/{NUM_QUESTIONS}')
print('Raw responses:', RAW_PATH)

## 6. Apply official ReasonIF checkers and compute dual IFS

- **`official_instruction_following`**: Evaluated on the complete reasoning trace (`forced_prefix + continuation`).
- **`continuation_instruction_following`**: Evaluated strictly on base Qwen's continuation (excluding the forced 10 tokens).

In [ ]:
if str(REASONIF_ROOT) not in sys.path:
    sys.path.insert(0, str(REASONIF_ROOT))
from src.eval_utils import evaluate_instruction_following, extract_final_answer
import src.instructions.instruction_checker as reasonif_instruction_checker
from fast_langdetect import detect as installed_language_detect

def compatible_detect(text, low_memory=False):
    try:
        result = installed_language_detect(text, low_memory=low_memory)
    except TypeError:
        result = installed_language_detect(text, model='lite' if low_memory else 'full')
    if isinstance(result, list):
        if not result:
            raise ValueError('fast-langdetect returned no predictions')
        return result[0]
    return result
reasonif_instruction_checker.detect = compatible_detect

def canonical_answer(value, source):
    value = str(value).strip().replace(',', '')
    if source in {'arc', 'gpqa'}:
        match = re.search(r'[ABCD]', value.upper())
        return match.group(0) if match else value.upper()
    try:
        number = float(value)
        return str(int(number)) if number.is_integer() else f'{number:.12g}'
    except ValueError:
        return value

records = [read_jsonl(RAW_PATH)[index] for index in range(NUM_QUESTIONS)]
assert len(records) == NUM_QUESTIONS, f'Generation incomplete: {len(records)}/{NUM_QUESTIONS}'
scored_rows = []
for record in records:
    full_reasoning = record['reasoning_content']
    continuation_reasoning = record['continuation_reasoning_content']
    
    # 1. Official IFS: scored over the entire reasoning trace (prefix + continuation)
    official_follows = all(evaluate_instruction_following(
        instruction_id_list=record['constraint_name'], parameters=record['constraint_args'],
        prompt=record['question'], response=full_reasoning,
    )) if full_reasoning.strip() else False
    
    # 2. Continuation IFS: scored strictly over base Qwen's generated continuation
    continuation_follows = all(evaluate_instruction_following(
        instruction_id_list=record['constraint_name'], parameters=record['constraint_args'],
        prompt=record['question'], response=continuation_reasoning,
    )) if continuation_reasoning.strip() else False
    
    predicted = extract_final_answer(record['content'], record['source'])
    correct = canonical_answer(predicted, record['source']) == canonical_answer(record['answer'], record['source'])
    has_answer_tags = '<answer>' in record['content'] and '</answer>' in record['content']
    has_reasoning_end = '</think>' in record['raw_continuation']
    
    scored_rows.append({
        **record, 'constraint': record['constraint_name'][0], 'predicted_answer': predicted,
        'instruction_following': official_follows,
        'official_instruction_following': official_follows,
        'continuation_instruction_following': continuation_follows,
        'answer_correct': correct,
        'joint_success': official_follows and correct,
        'official_joint_success': official_follows and correct,
        'continuation_joint_success': continuation_follows and correct,
        'has_answer_tags': has_answer_tags,
        'has_reasoning_end': has_reasoning_end,
        'format_valid': has_answer_tags and has_reasoning_end,
        'missing_answer': not has_answer_tags,
    })
SCORED_PATH = RESULTS_ROOT / 'scored_responses.jsonl'
SCORED_PATH.write_text(''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in scored_rows), encoding='utf-8')
print('Scored responses:', SCORED_PATH)


## 7. Paper-ready overall, constraint, and source results + McNemar statistical comparison

We aggregate all primary metrics and evaluate statistical significance against untouched baseline `Qwen3-14B Base`.

In [ ]:
import math
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from scipy import stats

frame = pd.DataFrame(scored_rows).sort_values('dataset_index').reset_index(drop=True)
metric_columns = [
    'official_instruction_following', 'continuation_instruction_following',
    'answer_correct', 'official_joint_success', 'continuation_joint_success',
    'format_valid', 'missing_answer', 'truncated'
]

def wilson_interval(successes, total, z=1.959963984540054):
    if total == 0:
        return float('nan'), float('nan')
    p = successes / total
    denominator = 1 + z*z/total
    center = (p + z*z/(2*total)) / denominator
    margin = z * math.sqrt(p*(1-p)/total + z*z/(4*total*total)) / denominator
    return center - margin, center + margin

long_rows = []
for metric in metric_columns:
    successes = int(frame[metric].sum())
    low, high = wilson_interval(successes, len(frame))
    long_rows.append({
        'model': MODEL_LABEL, 'metric': metric, 'successes': successes,
        'questions': len(frame), 'rate': successes/len(frame),
        'ci95_low': low, 'ci95_high': high
    })
overall_long = pd.DataFrame(long_rows)

overall = pd.DataFrame([{
    'model': MODEL_LABEL, 'questions': len(frame),
    **{column: frame[column].mean() for column in metric_columns},
    'mean_output_tokens': frame['output_tokens'].mean(),
    'median_output_tokens': frame['output_tokens'].median(),
}])

category = frame.groupby('constraint', sort=False).agg(
    questions=('dataset_index', 'count'),
    official_ifs=('official_instruction_following', 'mean'),
    continuation_ifs=('continuation_instruction_following', 'mean'),
    answer_accuracy=('answer_correct', 'mean'),
    official_joint=('official_joint_success', 'mean'),
    continuation_joint=('continuation_joint_success', 'mean'),
    format_valid=('format_valid', 'mean'),
    missing_answer=('missing_answer', 'mean'),
    truncated=('truncated', 'mean'),
    mean_output_tokens=('output_tokens', 'mean'),
    median_output_tokens=('output_tokens', 'median'),
).reset_index()

source_analysis = frame.groupby('source', sort=False).agg(
    questions=('dataset_index', 'count'),
    official_ifs=('official_instruction_following', 'mean'),
    continuation_ifs=('continuation_instruction_following', 'mean'),
    answer_accuracy=('answer_correct', 'mean'),
    official_joint=('official_joint_success', 'mean'),
    continuation_joint=('continuation_joint_success', 'mean'),
    format_valid=('format_valid', 'mean'),
    missing_answer=('missing_answer', 'mean'),
    truncated=('truncated', 'mean'),
    mean_output_tokens=('output_tokens', 'mean'),
    median_output_tokens=('output_tokens', 'median'),
).reset_index()

display(overall)
display(overall_long.style.format({'rate': '{:.1%}', 'ci95_low': '{:.1%}', 'ci95_high': '{:.1%}'}))
print('By ReasonIF constraint'); display(category)
print('By source task'); display(source_analysis)

overall.to_csv(RESULTS_ROOT / 'overall.csv', index=False)
overall.to_json(RESULTS_ROOT / 'overall.json', orient='records', indent=2)
overall_long.to_csv(RESULTS_ROOT / 'overall_metrics_with_ci95.csv', index=False)
category.to_csv(RESULTS_ROOT / 'by_constraint.csv', index=False)
source_analysis.to_csv(RESULTS_ROOT / 'by_source.csv', index=False)

# --- Paired McNemar comparison against untouched base model ---
BASE_PATH = LAUNCH_DIR / 'outputs' / 'reasonif_qwen3_14b_base' / 'scored_responses.jsonl'
if BASE_PATH.is_file():
    with open(BASE_PATH, 'r', encoding='utf-8') as f:
        base_rows = [json.loads(l) for l in f if l.strip()]
    base_df = pd.DataFrame(base_rows).sort_values('dataset_index').reset_index(drop=True)
    
    b_ifs = base_df['instruction_following'].values.astype(bool)
    m_off_ifs = frame['official_instruction_following'].values.astype(bool)
    m_cont_ifs = frame['continuation_instruction_following'].values.astype(bool)
    b_acc = base_df['answer_correct'].values.astype(bool)
    m_acc = frame['answer_correct'].values.astype(bool)
    mutual_corr = b_acc & m_acc
    
    def run_mcnemar(series_a, series_b):
        # table: [[a=0,b=0, a=0,b=1], [a=1,b=0, a=1,b=1]]
        n01 = int((~series_a & series_b).sum())
        n10 = int((series_a & ~series_b).sum())
        if (n01 + n10) == 0:
            return 1.0, 0.0
        chi2 = (abs(n01 - n10) - 1)**2 / (n01 + n10) if (n01 + n10) >= 25 else None
        res = stats.binomtest(n01, n01 + n10, 0.5)
        return res.pvalue, chi2
    
    p_off, _ = run_mcnemar(b_ifs, m_off_ifs)
    p_cont, _ = run_mcnemar(b_ifs, m_cont_ifs)
    p_mut_off, _ = run_mcnemar(b_ifs[mutual_corr], m_off_ifs[mutual_corr])
    p_mut_cont, _ = run_mcnemar(b_ifs[mutual_corr], m_cont_ifs[mutual_corr])
    
    paired_comp = pd.DataFrame([
        {
            'Comparison': f'Base vs {MODEL_LABEL} (Official IFS)',
            'Base Rate': b_ifs.mean(),
            'Prefix Rate': m_off_ifs.mean(),
            'Delta': m_off_ifs.mean() - b_ifs.mean(),
            'McNemar p-val': p_off,
        },
        {
            'Comparison': f'Base vs {MODEL_LABEL} (Continuation IFS)',
            'Base Rate': b_ifs.mean(),
            'Prefix Rate': m_cont_ifs.mean(),
            'Delta': m_cont_ifs.mean() - b_ifs.mean(),
            'McNemar p-val': p_cont,
        },
        {
            'Comparison': f'Base vs {MODEL_LABEL} (Mutual-Correct Official IFS, n={mutual_corr.sum()})',
            'Base Rate': b_ifs[mutual_corr].mean(),
            'Prefix Rate': m_off_ifs[mutual_corr].mean(),
            'Delta': m_off_ifs[mutual_corr].mean() - b_ifs[mutual_corr].mean(),
            'McNemar p-val': p_mut_off,
        },
        {
            'Comparison': f'Base vs {MODEL_LABEL} (Mutual-Correct Continuation IFS, n={mutual_corr.sum()})',
            'Base Rate': b_ifs[mutual_corr].mean(),
            'Prefix Rate': m_cont_ifs[mutual_corr].mean(),
            'Delta': m_cont_ifs[mutual_corr].mean() - b_ifs[mutual_corr].mean(),
            'McNemar p-val': p_mut_cont,
        },
    ])
    print('\n=== PAIRED STATISTICAL COMPARISON AGAINST BASE QWEN ===')
    display(paired_comp.style.format({
        'Base Rate': '{:.1%}', 'Prefix Rate': '{:.1%}', 'Delta': '{:+.1%}', 'McNemar p-val': '{:.4f}'
    }))
    paired_comp.to_csv(RESULTS_ROOT / 'paired_mcnemar_comparison.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
category.set_index('constraint')[['official_ifs', 'continuation_ifs', 'answer_accuracy']].plot.bar(ax=axes[0])
axes[0].set(title='ReasonIF IFS and Accuracy by Constraint', xlabel='', ylabel='rate', ylim=(0, 1))
category.set_index('constraint')[['missing_answer', 'truncated']].plot.bar(ax=axes[1], color=['#d95f02', '#7570b3'])
axes[1].set(title='Failure modes by constraint', xlabel='', ylabel='rate', ylim=(0, 1))
for axis in axes:
    axis.grid(axis='y', alpha=.2); axis.tick_params(axis='x', rotation=35)
fig.tight_layout()
FIGURE_PATH = RESULTS_ROOT / 'reasonif_category_analysis.png'
fig.savefig(FIGURE_PATH, dpi=180, bbox_inches='tight')
plt.show()
print('Saved analysis and plot to:', RESULTS_ROOT)


## 8. Inspect sample responses and export archive

In [ ]:
print('=== SAMPLE GENERATIONS (FIRST 3 EXAMPLES) ===')
for i in range(min(3, len(scored_rows))):
    r = scored_rows[i]
    print(f'\n--- QUESTION {i} ({r["constraint"]}) ---')
    print(f'Forced prefix: {repr(r["forced_prefix_text"])}')
    print(f'Official IFS: {r["official_instruction_following"]}, Continuation IFS: {r["continuation_instruction_following"]}')
    print(f'Answer Correct: {r["answer_correct"]}, Predicted: {r["predicted_answer"]}, Expected: {r["answer"]}')
    print('First 200 chars of continuation reasoning:')
    print(r['continuation_reasoning_content'][:200])
